<a href="https://colab.research.google.com/github/chandrahashkumar/Named-Entity-Recognition-App/blob/main/ner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Named Entity Recognition App**

In [ ]:
# @title
# !pip install "transformers<5.0.0"
# !pip install gradio

In [ ]:
# @title
from google.colab import userdata
import os
os.environ['HF_API_KEY'] = userdata.get('HF_API_KEY')

In [ ]:
import gradio as gr
from transformers import pipeline

get_completion = pipeline("ner", model="dslim/bert-base-NER")

In [ ]:
def merge_tokens(tokens):
    merged_tokens = []
    for token in tokens:
        if merged_tokens and token['entity'].startswith('I-') and merged_tokens[-1]['entity'].endswith(token['entity'][2:]):
            # If current token continues the entity of the last one, merge them
            last_token = merged_tokens[-1]
            last_token['word'] += token['word'].replace('##', '')
            last_token['end'] = token['end']
            last_token['score'] = (last_token['score'] + token['score']) / 2
        else:
            # Otherwise, add the token to the list
            merged_tokens.append(token)

    return merged_tokens

def ner(input):
    output = get_completion(input)
    merged_tokens = merge_tokens(output)
    return {"text": input, "entities": merged_tokens}

gr.close_all()
demo = gr.Interface(fn=ner,
                    inputs=[gr.Textbox(label="Text to find entities", lines=2)],
                    outputs=[gr.HighlightedText(label="Text with entities")],
                    title="Named Entity Recognition",
                    description="Find entities using the `dslim/bert-base-NER` model under the hood!",
                    flagging_mode="never",
                    examples=["My name is Chandrahsh Kumar, I'm live in Bihar","Today, Sunday"])

demo.launch(share=True)

In [ ]:
gr.close_all()